In [1]:
from __future__ import annotations

import numpy as np
import pandas as pd

DATASET_REPO = "ibm-research/argument_quality_ranking_30k"
QUALITY_COLUMN = "WA"
LABELS = ["low", "medium", "high"]


def _split_path(split: str, dataset_repo: str = DATASET_REPO) -> str:
    return f"hf://datasets/{dataset_repo}/{split}.csv"


def load_splits(dataset_repo: str = DATASET_REPO) -> dict[str, pd.DataFrame]:
    """Load train/dev/test splits from the dataset repository."""
    return {
        split: pd.read_csv(_split_path(split, dataset_repo))
        for split in ("train", "dev", "test")
    }


def build_input_text(df: pd.DataFrame) -> pd.Series:
    """Build model input text as Topic + Stance + Argument."""
    stance_map = {1: "pro", -1: "con"}
    topic = df["topic"].fillna("").astype(str)
    argument = df["argument"].fillna("").astype(str)
    stance = df["stance_WA"].map(stance_map).fillna("unknown")
    return "Topic: " + topic + " [SEP] Stance: " + stance + " [SEP] Argument: " + argument


def fit_quality_thresholds(train_scores: pd.Series) -> tuple[float, float]:
    """Fit low/medium/high thresholds from train-score terciles."""
    low_upper, medium_upper = np.quantile(train_scores, [1 / 3, 2 / 3])
    return float(low_upper), float(medium_upper)


def scores_to_labels(scores: pd.Series, thresholds: tuple[float, float]) -> pd.Series:
    """Convert numeric quality scores to low/medium/high labels."""
    low_upper, medium_upper = thresholds
    bins = [-np.inf, low_upper, medium_upper, np.inf]
    labels = pd.cut(scores, bins=bins, labels=LABELS, include_lowest=True)
    return labels.astype(str)


def get_data(
    dataset_repo: str = DATASET_REPO,
    quality_column: str = QUALITY_COLUMN,
) -> tuple[dict[str, tuple[pd.Series, pd.Series]], tuple[float, float]]:
    """Return prepared split data as (X, y) pairs plus fitted thresholds."""
    splits = load_splits(dataset_repo)
    thresholds = fit_quality_thresholds(splits["train"][quality_column])

    prepared: dict[str, tuple[pd.Series, pd.Series]] = {}
    for split_name, df in splits.items():
        x = build_input_text(df)
        y = scores_to_labels(df[quality_column], thresholds)
        prepared[split_name] = (x, y)

    return prepared, thresholds

# Data Exploration & Cleaning

This notebook performs comprehensive data exploration and cleaning for the argument quality ranking dataset.

## 1. Load Data & Basic Overview

In [2]:
# Load all splits
splits = load_splits()

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
for split_name, df in splits.items():
    print(f"\n{split_name.upper()} split:")
    print(f"  Shape: {df.shape} (rows, columns)")
    print(f"  Columns: {list(df.columns)}")
    
print("\n" + "=" * 60)

/Users/viorelscortia/Documents/Code/ArguMentor/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DATASET OVERVIEW

TRAIN split:
  Shape: (20974, 7) (rows, columns)
  Columns: ['argument', 'topic', 'set', 'WA', 'MACE-P', 'stance_WA', 'stance_WA_conf']

DEV split:
  Shape: (3208, 7) (rows, columns)
  Columns: ['argument', 'topic', 'set', 'WA', 'MACE-P', 'stance_WA', 'stance_WA_conf']

TEST split:
  Shape: (6315, 7) (rows, columns)
  Columns: ['argument', 'topic', 'set', 'WA', 'MACE-P', 'stance_WA', 'stance_WA_conf']



## 2. Data Types & Column Information

In [3]:
print("DATA TYPES & INFO (TRAIN SPLIT)")
print("=" * 60)
train_df = splits["train"]
print("\nData Types:")
print(train_df.dtypes)
print("\nMemory Usage:")
print(f"{train_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("\nColumn Details:")
for col in train_df.columns:
    print(f"\n{col}:")
    print(f"  Type: {train_df[col].dtype}")
    print(f"  Non-null count: {train_df[col].notna().sum()}")
    print(f"  Null count: {train_df[col].isna().sum()}")
    print(f"  Unique values: {train_df[col].nunique()}")

DATA TYPES & INFO (TRAIN SPLIT)

Data Types:
argument              str
topic                 str
set                   str
WA                float64
MACE-P            float64
stance_WA           int64
stance_WA_conf    float64
dtype: object

Memory Usage:
6.55 MB

Column Details:

argument:
  Type: str
  Non-null count: 20974
  Null count: 0
  Unique values: 20973

topic:
  Type: str
  Non-null count: 20974
  Null count: 0
  Unique values: 49

set:
  Type: str
  Non-null count: 20974
  Null count: 0
  Unique values: 1

WA:
  Type: float64
  Non-null count: 20974
  Null count: 0
  Unique values: 15140

MACE-P:
  Type: float64
  Non-null count: 20974
  Null count: 0
  Unique values: 18708

stance_WA:
  Type: int64
  Non-null count: 20974
  Null count: 0
  Unique values: 2

stance_WA_conf:
  Type: float64
  Non-null count: 20974
  Null count: 0
  Unique values: 5755


## 3. Missing Values Analysis

In [4]:
print("MISSING VALUES ANALYSIS")
print("=" * 60)
for split_name, df in splits.items():
    print(f"\n{split_name.upper()} split:")
    missing = df.isnull().sum()
    missing_pct = (df.isnull().sum() / len(df)) * 100
    missing_df = pd.DataFrame({
        'Column': missing.index,
        'Missing Count': missing.values,
        'Missing %': missing_pct.values
    })
    missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)
    if len(missing_df) == 0:
        print("  ✓ No missing values found")
    else:
        print(missing_df.to_string(index=False))

MISSING VALUES ANALYSIS

TRAIN split:
  ✓ No missing values found

DEV split:
  ✓ No missing values found

TEST split:
  ✓ No missing values found


## 4. Summary Statistics

In [5]:
print("SUMMARY STATISTICS (TRAIN SPLIT)")
print("=" * 60)
print(train_df.describe().to_string())
print("\n" + "=" * 60)
print("\nNUMERIC COLUMNS STATISTICS:")
numeric_cols = train_df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    print(f"\n{col}:")
    print(f"  Min: {train_df[col].min()}")
    print(f"  Max: {train_df[col].max()}")
    print(f"  Mean: {train_df[col].mean():.4f}")
    print(f"  Median: {train_df[col].median():.4f}")
    print(f"  Std Dev: {train_df[col].std():.4f}")

SUMMARY STATISTICS (TRAIN SPLIT)
                 WA        MACE-P     stance_WA  stance_WA_conf
count  20974.000000  2.097400e+04  20974.000000    20974.000000
mean       0.793734  5.707986e-01      0.015352        0.953322
std        0.190417  3.680238e-01      0.999906        0.094553
min        0.000000  5.430000e-14     -1.000000        0.335360
25%        0.685525  1.792709e-01     -1.000000        0.926239
50%        0.842135  6.868957e-01      1.000000        1.000000
75%        0.948733  9.167528e-01      1.000000        1.000000
max        1.000000  9.999999e-01      1.000000        1.000000


NUMERIC COLUMNS STATISTICS:

WA:
  Min: 0.0
  Max: 1.0
  Mean: 0.7937
  Median: 0.8421
  Std Dev: 0.1904

MACE-P:
  Min: 5.43e-14
  Max: 0.999999897
  Mean: 0.5708
  Median: 0.6869
  Std Dev: 0.3680

stance_WA:
  Min: -1
  Max: 1
  Mean: 0.0154
  Median: 1.0000
  Std Dev: 0.9999

stance_WA_conf:
  Min: 0.335360073
  Max: 1.0
  Mean: 0.9533
  Median: 1.0000
  Std Dev: 0.0946


## 5. Target Variable & Class Distribution

In [6]:
print("QUALITY SCORE & CLASS DISTRIBUTION")
print("=" * 60)
for split_name, df in splits.items():
    print(f"\n{split_name.upper()} split ({len(df)} samples):")
    quality_col = QUALITY_COLUMN
    print(f"\n  {quality_col} Score Distribution:")
    print(df[quality_col].describe().to_string())
    print(f"\n  {quality_col} Score Range: [{df[quality_col].min()}, {df[quality_col].max()}]")
    
    # Check class distribution after labeling
    thresholds = fit_quality_thresholds(splits["train"][quality_col])
    labels = scores_to_labels(df[quality_col], thresholds)
    print(f"\n  Class Distribution (after thresholding):")
    class_dist = labels.value_counts().sort_index()
    for label, count in class_dist.items():
        pct = (count / len(labels)) * 100
        print(f"    {label:8s}: {count:6d} ({pct:5.2f}%)")
    print()

QUALITY SCORE & CLASS DISTRIBUTION

TRAIN split (20974 samples):

  WA Score Distribution:
count    20974.000000
mean         0.793734
std          0.190417
min          0.000000
25%          0.685525
50%          0.842135
75%          0.948733
max          1.000000

  WA Score Range: [0.0, 1.0]

  Class Distribution (after thresholding):
    high    :   6991 (33.33%)
    low     :   6992 (33.34%)
    medium  :   6991 (33.33%)


DEV split (3208 samples):

  WA Score Distribution:
count    3208.000000
mean        0.795556
std         0.193328
min         0.045923
25%         0.693082
50%         0.846339
75%         0.947002
max         1.000000

  WA Score Range: [0.045923473, 1.0]

  Class Distribution (after thresholding):
    high    :   1100 (34.29%)
    low     :   1028 (32.04%)
    medium  :   1080 (33.67%)


TEST split (6315 samples):

  WA Score Distribution:
count    6315.000000
mean        0.781193
std         0.198946
min         0.000000
25%         0.668542
50%         0.8

## 6. Categorical Variables Analysis

In [7]:
print("CATEGORICAL VARIABLES ANALYSIS (TRAIN SPLIT)")
print("=" * 60)
categorical_cols = train_df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    print(f"\n{col}:")
    print(f"  Unique values: {train_df[col].nunique()}")
    print(f"  Top 10 values:")
    top_vals = train_df[col].value_counts().head(10)
    for val, count in top_vals.items():
        pct = (count / len(train_df)) * 100
        print(f"    {str(val)[:50]:50s}: {count:6d} ({pct:5.2f}%)")

CATEGORICAL VARIABLES ANALYSIS (TRAIN SPLIT)

argument:
  Unique values: 20973
  Top 10 values:
    The true purposes of telemarketing are to pressure:      2 ( 0.01%)
    "marriage" isn't keeping up with the times.  aband:      1 ( 0.00%)
    .a multi-party system would be too confusing and g:      1 ( 0.00%)
    `people reach their limit when it comes to their q:      1 ( 0.00%)
    100% agree, should they do that, it would be a goo:      1 ( 0.00%)
    A ban on naturopathy creates a cohesive front betw:      1 ( 0.00%)
    A ban would be disasterous for the surrogate mothe:      1 ( 0.00%)
    A ban would be inffective, people who want a baby :      1 ( 0.00%)
    a ban would only increase the already concerning t:      1 ( 0.00%)
    A basic principle of punishment is that it should :      1 ( 0.00%)

topic:
  Unique values: 49
  Top 10 values:
    We should fight for the abolition of nuclear weapo:    554 ( 2.64%)
    We should ban naturopathy                         :    540 ( 2.

/var/folders/lp/4vbz2pkn62n5gbqhn5nxcggm0000gn/T/ipykernel_48497/710245703.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = train_df.select_dtypes(include=['object']).columns


## 7. Text Data Quality Analysis

In [8]:
print("TEXT DATA QUALITY ANALYSIS (TRAIN SPLIT)")
print("=" * 60)
text_cols = ['topic', 'argument']
for col in text_cols:
    if col not in train_df.columns:
        continue
    print(f"\n{col}:")
    
    # Empty strings
    empty_count = (train_df[col].fillna('').str.len() == 0).sum()
    print(f"  Empty strings: {empty_count} ({(empty_count/len(train_df)*100):.2f}%)")
    
    # Length statistics
    lengths = train_df[col].fillna('').str.len()
    print(f"  Length statistics:")
    print(f"    Min: {lengths.min()}")
    print(f"    Max: {lengths.max()}")
    print(f"    Mean: {lengths.mean():.2f}")
    print(f"    Median: {lengths.median():.2f}")
    
    # Very short texts (< 5 chars)
    short_count = (lengths < 5).sum()
    print(f"  Very short texts (< 5 chars): {short_count} ({(short_count/len(train_df)*100):.2f}%)")
    
    # Duplicates
    unique_count = train_df[col].nunique()
    print(f"  Unique values: {unique_count}/{len(train_df)} ({(unique_count/len(train_df)*100):.2f}%)")

TEXT DATA QUALITY ANALYSIS (TRAIN SPLIT)

topic:
  Empty strings: 0 (0.00%)
  Length statistics:
    Min: 21
    Max: 52
    Mean: 34.37
    Median: 33.00
  Very short texts (< 5 chars): 0 (0.00%)
  Unique values: 49/20974 (0.23%)

argument:
  Empty strings: 0 (0.00%)
  Length statistics:
    Min: 35
    Max: 245
    Mean: 107.69
    Median: 101.00
  Very short texts (< 5 chars): 0 (0.00%)
  Unique values: 20973/20974 (100.00%)


## 8. Duplicate Records Analysis

In [9]:
print("DUPLICATE RECORDS ANALYSIS")
print("=" * 60)
for split_name, df in splits.items():
    print(f"\n{split_name.upper()} split:")
    
    # Full row duplicates
    full_dupes = df.duplicated().sum()
    print(f"  Full row duplicates: {full_dupes} ({(full_dupes/len(df)*100):.2f}%)")
    
    # Duplicates by topic + argument
    if 'topic' in df.columns and 'argument' in df.columns:
        subset_dupes = df.duplicated(subset=['topic', 'argument']).sum()
        print(f"  Duplicates by (topic, argument): {subset_dupes} ({(subset_dupes/len(df)*100):.2f}%)")
    
    # Duplicates by argument only
    if 'argument' in df.columns:
        arg_dupes = df.duplicated(subset=['argument']).sum()
        print(f"  Duplicates by argument: {arg_dupes} ({(arg_dupes/len(df)*100):.2f}%)")

DUPLICATE RECORDS ANALYSIS

TRAIN split:
  Full row duplicates: 0 (0.00%)
  Duplicates by (topic, argument): 1 (0.00%)
  Duplicates by argument: 1 (0.00%)

DEV split:
  Full row duplicates: 0 (0.00%)
  Duplicates by (topic, argument): 0 (0.00%)
  Duplicates by argument: 0 (0.00%)

TEST split:
  Full row duplicates: 0 (0.00%)
  Duplicates by (topic, argument): 0 (0.00%)
  Duplicates by argument: 0 (0.00%)


## 9. Data Quality Issues Summary

In [10]:
print("DATA QUALITY ISSUES SUMMARY")
print("=" * 60)

issues = {
    "Missing Values": [],
    "Empty Strings": [],
    "Duplicates": [],
    "Class Imbalance": [],
    "Outliers": []
}

# Check for missing values
for split_name, df in splits.items():
    missing = df.isnull().sum()
    if missing.sum() > 0:
        issues["Missing Values"].append(f"{split_name}: {missing.sum()} missing cells")

# Check for empty strings in text columns
for split_name, df in splits.items():
    for col in ['topic', 'argument']:
        if col in df.columns:
            empty = (df[col].fillna('').str.len() == 0).sum()
            if empty > 0:
                issues["Empty Strings"].append(f"{split_name}.{col}: {empty} empty strings")

# Check for duplicates
for split_name, df in splits.items():
    dupes = df.duplicated().sum()
    if dupes > 0:
        issues["Duplicates"].append(f"{split_name}: {dupes} full duplicates")

# Check class imbalance
train_df = splits["train"]
thresholds = fit_quality_thresholds(train_df[QUALITY_COLUMN])
labels = scores_to_labels(train_df[QUALITY_COLUMN], thresholds)
class_dist = labels.value_counts()
min_class = class_dist.min()
max_class = class_dist.max()
imbalance_ratio = max_class / min_class
if imbalance_ratio > 1.5:
    issues["Class Imbalance"].append(f"Imbalance ratio: {imbalance_ratio:.2f}x")

# Check for outliers in numeric columns
for col in train_df.select_dtypes(include=[np.number]).columns:
    Q1 = train_df[col].quantile(0.25)
    Q3 = train_df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((train_df[col] < Q1 - 1.5 * IQR) | (train_df[col] > Q3 + 1.5 * IQR)).sum()
    if outliers > 0:
        issues["Outliers"].append(f"{col}: {outliers} outliers detected")

# Print summary
print("\n✓ IDENTIFIED ISSUES:\n")
has_issues = False
for category, items in issues.items():
    if items:
        has_issues = True
        print(f"  {category}:")
        for item in items:
            print(f"    • {item}")

if not has_issues:
    print("  ✓ No significant quality issues detected!")
    
print("\n" + "=" * 60)

DATA QUALITY ISSUES SUMMARY

✓ IDENTIFIED ISSUES:

  Outliers:
    • WA: 371 outliers detected
    • stance_WA_conf: 1896 outliers detected



## 10. Data Cleaning Implementation

In [12]:
def clean_data(
    df: pd.DataFrame,
    remove_empty_argument: bool = True,
    min_argument_length: int = 5,
    remove_duplicates: bool = True,
    remove_null_quality: bool = True,
) -> pd.DataFrame:
    """
    Clean dataset based on identified issues.
    
    Args:
        df: Input dataframe
        remove_empty_argument: Remove rows with empty arguments
        min_argument_length: Minimum allowed argument length
        remove_duplicates: Remove exact duplicate rows
        remove_null_quality: Remove rows missing quality score
    
    Returns:
        Cleaned dataframe with cleaning statistics
    """
    original_len = len(df)
    df_clean = df.copy()
    
    stats = {"Original": original_len}
    
    # Remove rows with null quality score
    if remove_null_quality and QUALITY_COLUMN in df_clean.columns:
        df_clean = df_clean[df_clean[QUALITY_COLUMN].notna()]
        stats["After removing null quality"] = len(df_clean)
    
    # Remove empty arguments
    if remove_empty_argument and 'argument' in df_clean.columns:
        df_clean = df_clean[df_clean['argument'].fillna('').str.len() > 0]
        stats["After removing empty arguments"] = len(df_clean)
    
    # Remove short arguments
    if min_argument_length > 0 and 'argument' in df_clean.columns:
        df_clean = df_clean[df_clean['argument'].fillna('').str.len() >= min_argument_length]
        stats["After removing short arguments"] = len(df_clean)
    
    # Remove exact duplicates
    if remove_duplicates:
        df_clean = df_clean.drop_duplicates()
        stats["After removing duplicates"] = len(df_clean)
    
    # Strip whitespace from text columns
    text_cols = df_clean.select_dtypes(include=['object']).columns
    for col in text_cols:
        df_clean[col] = df_clean[col].apply(
            lambda x: x.strip() if isinstance(x, str) else x
        )
    
    stats["Final"] = len(df_clean)
    stats["Rows removed"] = original_len - len(df_clean)
    stats["Percentage retained"] = f"{(len(df_clean)/original_len*100):.2f}%"
    
    return df_clean, stats


# Apply cleaning to all splits
print("APPLYING DATA CLEANING")
print("=" * 60)
cleaned_splits = {}
for split_name, df in splits.items():
    print(f"\nCleaning {split_name} split:")
    cleaned_df, stats = clean_data(df)
    cleaned_splits[split_name] = cleaned_df
    
    for key, val in stats.items():
        print(f"  {key}: {val}")

print("\n" + "=" * 60)
print("\n✓ Data cleaning complete!")

APPLYING DATA CLEANING

Cleaning train split:
  Original: 20974
  After removing null quality: 20974
  After removing empty arguments: 20974
  After removing short arguments: 20974
  After removing duplicates: 20974
  Final: 20974
  Rows removed: 0
  Percentage retained: 100.00%

Cleaning dev split:
  Original: 3208
  After removing null quality: 3208
  After removing empty arguments: 3208
  After removing short arguments: 3208
  After removing duplicates: 3208
  Final: 3208
  Rows removed: 0
  Percentage retained: 100.00%

Cleaning test split:
  Original: 6315
  After removing null quality: 6315
  After removing empty arguments: 6315
  After removing short arguments: 6315
  After removing duplicates: 6315
  Final: 6315
  Rows removed: 0
  Percentage retained: 100.00%


✓ Data cleaning complete!


/var/folders/lp/4vbz2pkn62n5gbqhn5nxcggm0000gn/T/ipykernel_48497/2854024606.py:47: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df_clean.select_dtypes(include=['object']).columns
/var/folders/lp/4vbz2pkn62n5gbqhn5nxcggm0000gn/T/ipykernel_48497/2854024606.py:47: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See ht